
> **Input tokens** are converted to vectors via embedding + positional encoding, giving each token both meaning and position.
> **Multi-head self-attention** lets every token look at every other token simultaneously, building rich contextual understanding.
> **A feed-forward network + residual connections + layer norm** then refines each token's representation — this block repeats N times to produce the final output.

In [1]:
import torch
import torch.nn as nn
import math


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # Learned projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        B, T, _ = x.size()
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B, T, _ = x.size()

        # Project inputs to Q, K, V and split into heads
        Q = self.split_heads(self.W_q(x))   # (B, H, T, d_k)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / self.scale  # (B, H, T, T)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = attn_weights @ V                            # (B, H, T, d_k)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)  # merge heads

        return self.W_o(out)


class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),             # GELU used in BERT/ViT; swap for ReLU if desired
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_seq_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # Pre-compute the sine/cosine positional matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices
        pe = pe.unsqueeze(0)                           # (1, max_seq_len, d_model)
        self.register_buffer('pe', pe)                 # not a learnable parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class EncoderLayer(nn.Module):
    """One encoder layer = self-attention sublayer + FFN sublayer, each with residual + LayerNorm."""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn  = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ffn        = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # Sublayer 1: multi-head self-attention with residual connection
        x = self.norm1(x + self.dropout(self.self_attn(x, mask)))

        # Sublayer 2: feed-forward network with residual connection
        x = self.norm2(x + self.dropout(self.ffn(x)))

        return x


class TransformerEncoder(nn.Module):
    """Full transformer encoder: embedding → positional encoding → N × EncoderLayer."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int   = 512,
        num_heads: int = 8,
        num_layers: int= 6,
        d_ff: int      = 2048,
        max_seq_len: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, max_seq_len, dropout)
        self.layers    = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)   # final layer norm (used in some variants)
        self.d_model = d_model

    def forward(
        self,
        token_ids: torch.Tensor,           # (batch, seq_len)
        mask: torch.Tensor = None,          # (batch, 1, 1, seq_len) padding mask
    ) -> torch.Tensor:
        # Scale embeddings before adding positional encoding (original paper)
        x = self.embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, mask)

        return self.norm(x)   # (batch, seq_len, d_model)


# ─────────────────────────── Quick demo ───────────────────────────
if __name__ == "__main__":
    VOCAB   = 30_000
    BATCH   = 2
    SEQ_LEN = 16

    model = TransformerEncoder(
        vocab_size  = VOCAB,
        d_model     = 512,
        num_heads   = 8,
        num_layers  = 6,
        d_ff        = 2048,
        max_seq_len = 512,
        dropout     = 0.1,
    )

    tokens = torch.randint(0, VOCAB, (BATCH, SEQ_LEN))
    output = model(tokens)

    print(f"Input  shape: {tokens.shape}")   # (2, 16)
    print(f"Output shape: {output.shape}")   # (2, 16, 512)
    print(f"Parameters:   {sum(p.numel() for p in model.parameters()):,}")

Input  shape: torch.Size([2, 16])
Output shape: torch.Size([2, 16, 512])
Parameters:   34,275,328
